# Stock Market Prediction with an LSTM

This notebook downloads daily market data, explores price behavior, trains a leakage-safe Long Short-Term Memory (LSTM) model, evaluates it on a chronological hold-out set, and forecasts future closing prices. Forecasts are educational estimates and are not financial advice.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import yfinance as yf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, LSTM

np.random.seed(42)
    

The imports cover data acquisition, numerical analysis, visualization, statistical checks, preprocessing, and the Keras LSTM model. The fixed seed makes the experiment easier to reproduce.

In [ ]:
TICKER = "AAPL"
START = "2018-01-01"
END = pd.Timestamp.today().strftime("%Y-%m-%d")
LOOK_BACK = 60
TRAIN_FRACTION = 0.8

data = yf.download(TICKER, start=START, end=END, auto_adjust=False, progress=False)
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)
data = data[["Open", "High", "Low", "Close", "Volume"]].dropna().sort_index()
if data.empty:
    raise ValueError(f"No data was returned for {TICKER}.")
data.head(), data.shape


The dataset contains daily OHLCV observations. Missing rows are removed only after checking that the required market columns exist; the chronological index is preserved because order is essential for forecasting.

In [ ]:
display(data.describe().T)
display(data.isna().sum().to_frame("missing_values"))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(data.index, data["Close"], color="tab:blue")
axes[0].set_title(f"{TICKER} closing price")
axes[0].set_ylabel("Price")
axes[1].plot(data.index, data["Volume"], color="tab:green")
axes[1].set_title("Daily trading volume")
axes[1].set_ylabel("Volume")
plt.tight_layout()
plt.show()


The summary and chart show the scale, spread, and long-term trend of the target. Volume is displayed separately because its magnitude is not comparable to price. Any remaining missing values would need treatment before modeling.

In [ ]:
returns = data["Close"].pct_change().dropna()
print(f"Average daily return: {returns.mean():.4%}")
print(f"Daily return volatility: {returns.std():.4%}")
print(f"Return skewness: {stats.skew(returns):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(returns, bins=60, kde=True, ax=axes[0], color="tab:purple")
axes[0].set_title("Distribution of daily returns")
axes[0].set_xlabel("Return")
axes[1].plot(returns.index, returns, color="tab:red", linewidth=0.7)
axes[1].set_title("Daily returns over time")
axes[1].set_ylabel("Return")
plt.tight_layout()
plt.show()


Returns are used to inspect volatility rather than as the model target. Large fluctuations and non-normal behavior are common in financial series, which is why the LSTM is evaluated on unseen dates instead of relying only on training loss.

In [ ]:
close_values = data["Close"].to_numpy(dtype="float32").reshape(-1, 1)
split_index = int(len(close_values) * TRAIN_FRACTION)
scaler = MinMaxScaler()
scaler.fit(close_values[:split_index])
scaled_values = scaler.transform(close_values)

def make_sequences(values, window):
    if len(values) <= window:
        raise ValueError(f"At least {window + 1} rows are required.")
    x = np.array([values[i - window:i] for i in range(window, len(values))])
    y = np.array([values[i] for i in range(window, len(values))])
    return x.reshape(-1, window, 1), y

X, y = make_sequences(scaled_values, LOOK_BACK)
target_indices = np.arange(LOOK_BACK, len(close_values))
train_mask = target_indices < split_index
test_mask = ~train_mask
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]
print(f"Training sequences: {len(X_train):,}; test sequences: {len(X_test):,}")


The chronological split happens before fitting the scaler, preventing future prices from influencing preprocessing. Test sequences are allowed to use the preceding look-back observations because those observations would be available at prediction time.

In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(LOOK_BACK, 1)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1),
])
model.compile(optimizer="adam", loss="mean_squared_error")
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=30,
    batch_size=32,
    callbacks=[EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)],
    verbose=1,
)

plt.figure(figsize=(10, 4))
plt.plot(history.history["loss"], label="Training loss", color="tab:blue")
plt.plot(history.history["val_loss"], label="Validation loss", color="tab:orange")
plt.title("LSTM training history")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


The two LSTM layers learn patterns across the look-back window, while dropout reduces reliance on individual activations. Early stopping restores the best validation checkpoint to limit overfitting.

In [ ]:
test_predictions = scaler.inverse_transform(model.predict(X_test, verbose=0)).ravel()
test_actual = scaler.inverse_transform(y_test.reshape(-1, 1)).ravel()
test_dates = data.index[target_indices[test_mask]]
mae = mean_absolute_error(test_actual, test_predictions)
rmse = np.sqrt(mean_squared_error(test_actual, test_predictions))
print(f"Test MAE: ${mae:,.2f}")
print(f"Test RMSE: ${rmse:,.2f}")

evaluation = pd.DataFrame({"Actual": test_actual, "Predicted": test_predictions}, index=test_dates)
plt.figure(figsize=(14, 5))
plt.plot(evaluation.index, evaluation["Actual"], label="Actual", color="tab:blue")
plt.plot(evaluation.index, evaluation["Predicted"], label="Predicted", color="tab:orange")
plt.title(f"{TICKER} held-out test predictions")
plt.xlabel("Date")
plt.ylabel("Close price")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


MAE is the average absolute dollar error, while RMSE penalizes larger misses more heavily. The test chart shows whether predictions follow the direction and level of prices on dates that were not used for fitting.

In [ ]:
horizon = 5
recent_scaled = scaled_values.ravel().tolist()
future_predictions = []
for _ in range(horizon):
    model_input = np.array(recent_scaled[-LOOK_BACK:]).reshape(1, LOOK_BACK, 1)
    next_scaled = float(model.predict(model_input, verbose=0)[0, 0])
    recent_scaled.append(next_scaled)
    future_predictions.append(float(scaler.inverse_transform([[next_scaled]])[0, 0]))

future_dates = pd.bdate_range(data.index[-1] + pd.Timedelta(days=1), periods=horizon)
forecast = pd.DataFrame({"Predicted Close": future_predictions}, index=future_dates)
display(forecast.style.format("${:,.2f}"))
last_close = float(data["Close"].iloc[-1])
next_close = future_predictions[0]
direction = "UP" if next_close >= last_close else "DOWN"
print(f"Next predicted movement: {direction} ({(next_close - last_close) / last_close:.2%})")


The forecast is recursive: each predicted close becomes part of the next input window. This supports a multi-day horizon, but uncertainty compounds with each step, so the first forecast is generally more reliable than later values.

### Q&A

- **Can the LSTM predict the next movement and price?** Yes. It produces a next-close estimate and labels the movement as UP or DOWN relative to the latest close.
- **How should performance be judged?** Use the held-out chronological MAE and RMSE, together with the actual-versus-predicted chart. A lower error is useful, but it does not guarantee profitable trading.

### Data Analysis Key Findings

- The notebook reports the number of observations, missing values, daily-return statistics, and test-set error for the selected ticker and date range.
- The chronological split and training-only scaler prevent future observations from leaking into model training.
- The test plot provides a visual check of where the LSTM tracks or misses the observed closing-price trend.

### Insights or Next Steps

- Compare this LSTM with a naive last-price baseline and alternative features such as returns, moving averages, and volume before using it operationally.
- Paper-test any movement strategy with transaction costs and strict risk controls; this model is not a guarantee of future returns.
